# Unidade 3 — Aula 4
## Pipeline de pré-processamento, preparação para modelagem e deploy


**Situação:** toda semana o garçom envia ao dono do restaurante uma planilha nova de gorjetas. Repetir à mão todos os passos das últimas aulas não é viável. Nesta atividade, você vai transformar esses passos em um **pipeline**: uma sequência fixa de funções que leva o dado bruto ao dado pronto, sempre na mesma ordem.

**Base de dados:** *Tips* (BRYANT, P. G.; SMITH, M. A., 1995), carregada de uma cópia pública:
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv

As 200 primeiras linhas fazem o papel do **histórico** do restaurante; as 44 restantes, da **semana nova**.

Aluno: Carlos Gabriel Raposo Landim

## 1. Carga e divisão entre histórico e semana nova

In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
from pathlib import Path
bruto = pd.read_csv(Path('dados/tips.csv') if Path('dados/tips.csv').exists() else URL)
historico_bruto = bruto.iloc[:200]
semana_bruta = bruto.iloc[200:]
print(len(historico_bruto), len(semana_bruta))

200 44


## 2. Passo 1 do pipeline: limpeza
**Exemplo:** função que renomeia as colunas e remove duplicatas.

In [2]:
def limpar(df):
    df = df.copy()
    df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
    return df.drop_duplicates()


historico_limpo = limpar(historico_bruto)
print("Linhas removidas no histórico:", len(historico_bruto) - len(historico_limpo))
historico_limpo.head()

Linhas removidas no histórico: 0


,conta,gorjeta,sexo,fumante,dia,turno,pessoas
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 3. Passo 2 do pipeline: criação de atributos
**Exemplo:** função com os dois primeiros atributos da aula anterior.

In [3]:
def criar_atributos_exemplo(df):
    df = df.copy()
    df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
    df["conta_por_pessoa"] = df["conta"] / df["pessoas"]
    return df


criar_atributos_exemplo(historico_limpo).head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas,percentual_gorjeta,conta_por_pessoa
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500


**Sua vez 3.1:** escreva a função `criar_atributos(df)`, com os dois atributos do exemplo e mais três: `fim_de_semana` (1 para sábado e domingo), `grupo_grande` (1 para 4 pessoas ou mais) e `faixa_conta` (até 15, de 15 a 25 e acima de 25 dólares, com os rótulos baixa, média e alta).

In [4]:
def criar_atributos(df):
    df = df.copy()
    df['percentual_gorjeta'] = df['gorjeta'] / df['conta'] * 100
    df['conta_por_pessoa'] = df['conta'] / df['pessoas']
    df['fim_de_semana'] = df['dia'].isin(['Sat', 'Sun']).astype(int)
    df['grupo_grande'] = (df['pessoas'] >= 4).astype(int)
    df['faixa_conta'] = pd.cut(df['conta'], bins=[0, 15, 25, np.inf], labels=['baixa', 'média', 'alta'])
    return df

criar_atributos(historico_limpo).head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas,percentual_gorjeta,conta_por_pessoa,fim_de_semana,grupo_grande,faixa_conta
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,1,0,média
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,1,0,baixa
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,1,0,média
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,1,0,média
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,1,1,média


## 4. Passo 3 do pipeline: codificação
**Exemplo:** codificação binária de `sexo` e `fumante` e codificação ordinal de `faixa_conta`.

In [5]:
def codificar_exemplo(df):
    df = df.copy()
    df["sexo"] = (df["sexo"] == "Female").astype(int)
    df["fumante"] = (df["fumante"] == "Yes").astype(int)
    df["faixa_conta"] = df["faixa_conta"].map({"baixa": 0, "média": 1, "alta": 2}).astype(int)
    return df


codificar_exemplo(criar_atributos(historico_limpo)).head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas,percentual_gorjeta,conta_por_pessoa,fim_de_semana,grupo_grande,faixa_conta
0,16.99,1.01,1,0,Sun,Dinner,2,5.944673,8.495000,1,0,1
1,10.34,1.66,0,0,Sun,Dinner,3,16.054159,3.446667,1,0,0
2,21.01,3.50,0,0,Sun,Dinner,3,16.658734,7.003333,1,0,1
3,23.68,3.31,0,0,Sun,Dinner,2,13.978041,11.840000,1,0,1
4,24.59,3.61,1,0,Sun,Dinner,4,14.680765,6.147500,1,1,1


**Sua vez 4.1:** escreva a função `codificar(df)`, com as codificações do exemplo e mais duas: `turno` binário (1 para jantar) e one-hot de `dia` com `pd.get_dummies(..., prefix="dia", dtype=int)`. Aplique a função ao histórico e à semana nova e compare as colunas das duas bases.

In [6]:
def codificar(df):
    df = df.copy()
    df['sexo'] = (df['sexo'] == 'Female').astype(int)
    df['fumante'] = (df['fumante'] == 'Yes').astype(int)
    df['faixa_conta'] = df['faixa_conta'].map({'baixa': 0, 'média': 1, 'alta': 2}).astype(int)
    df['turno'] = (df['turno'] == 'Dinner').astype(int)
    return pd.get_dummies(df, columns=['dia'], prefix='dia', dtype=int)

hist_cod = codificar(criar_atributos(limpar(historico_bruto)))
sem_cod = codificar(criar_atributos(limpar(semana_bruta)))
print('Histórico:', list(hist_cod.columns))
print('Semana:', list(sem_cod.columns))
print('Colunas idênticas:', list(hist_cod.columns) == list(sem_cod.columns))

Histórico: ['conta', 'gorjeta', 'sexo', 'fumante', 'turno', 'pessoas', 'percentual_gorjeta', 'conta_por_pessoa', 'fim_de_semana', 'grupo_grande', 'faixa_conta', 'dia_Fri', 'dia_Sat', 'dia_Sun', 'dia_Thur']
Semana: ['conta', 'gorjeta', 'sexo', 'fumante', 'turno', 'pessoas', 'percentual_gorjeta', 'conta_por_pessoa', 'fim_de_semana', 'grupo_grande', 'faixa_conta', 'dia_Fri', 'dia_Sat', 'dia_Thur']
Colunas idênticas: False


**Pergunta 4.2:** as duas bases ficaram com as mesmas colunas? Por quê? Que problema isso causaria se a semana nova fosse enviada para a próxima etapa do projeto?

**Resposta:**

Não. O histórico contém registros de domingo e cria a coluna `dia_Sun`; as 44 linhas da semana nova não contêm domingo. O get_dummies aplicado separadamente produz esquemas diferentes. Uma próxima etapa que espera as mesmas variáveis pode falhar ou associar posições às colunas erradas. O reindex com as colunas do histórico mantém o esquema e preenche a ausência dessa categoria com zero.

**Exemplo:** fixar as colunas da semana nova a partir das colunas do histórico; categorias ausentes recebem 0.

In [7]:
sem_cod_alinhada = sem_cod.reindex(columns=hist_cod.columns, fill_value=0)
print(list(sem_cod_alinhada.columns) == list(hist_cod.columns))
sem_cod_alinhada.filter(like="dia_").sum()

True


dia_Fri      7
dia_Sat     30
dia_Sun      0
dia_Thur     7
dtype: int64

## 5. Passo 4 do pipeline: a receita de padronização
A receita (médias e desvios) é calculada **uma única vez, no histórico**, e depois aplicada a qualquer base nova.

**Exemplo:** funções para aprender e aplicar a receita.

In [8]:
COLUNAS_PADRONIZAR = ["conta", "percentual_gorjeta", "conta_por_pessoa"]


def aprender_receita(historico):
    return {
        "colunas": list(historico.columns),
        "media": historico[COLUNAS_PADRONIZAR].mean(),
        "desvio": historico[COLUNAS_PADRONIZAR].std(),
    }


def aplicar_receita(df, receita):
    df = df.reindex(columns=receita["colunas"], fill_value=0).copy()
    df[COLUNAS_PADRONIZAR] = (df[COLUNAS_PADRONIZAR] - receita["media"]) / receita["desvio"]
    return df


receita = aprender_receita(hist_cod)
receita["media"].round(2)

conta                 19.81
percentual_gorjeta    16.26
conta_por_pessoa       7.92
dtype: float64

**Sua vez 5.1:** padronize o percentual da gorjeta da semana nova de duas formas: (a) com a receita do histórico e (b) com a média e o desvio da própria semana. Compare o resultado para a mesa de índice 214 e a média de cada versão.

In [9]:
percentual_semana = sem_cod['percentual_gorjeta']
z_historico = (percentual_semana - receita['media']['percentual_gorjeta']) / receita['desvio']['percentual_gorjeta']
z_semana = (percentual_semana - percentual_semana.mean()) / percentual_semana.std()
display(pd.DataFrame({'indice_214': [z_historico.loc[214], z_semana.loc[214]], 'media_dos_z': [z_historico.mean(), z_semana.mean()]}, index=['Receita do histórico', 'Receita da própria semana']).round(4))
print('Média histórica:', round(hist_cod.percentual_gorjeta.mean(), 4))
print('Média da semana:', round(percentual_semana.mean(), 4))

,indice_214,media_dos_z
Receita do histórico,1.0978,-0.1649
Receita da própria semana,1.3881,-0.0000


Média histórica: 16.2648
Média da semana: 15.2416


**Pergunta 5.2:** com a média da própria semana, a média padronizada da semana é sempre 0. Por que isso impede o dono de perceber que a semana foi mais fraca que o histórico?

**Resposta:**

Recalcular a receita na semana força a média dos escores a zero e apaga o deslocamento em relação ao histórico. A média de gorjetas caiu de 16,2648% para 15,2416% (−1,0231 ponto percentual). Usando a referência histórica, a semana tem média padronizada −0,1649; usando a própria semana, fica próxima de zero. Para a mesa 214, os escores são 1,0978 e 1,3881. A referência fixa permite comparar períodos sem recalibrar a régua a cada semana.

## 6. O pipeline completo
**Exemplo:** função `preparar` que encadeia todos os passos.

In [10]:
def preparar(df_bruto, receita):
    df = limpar(df_bruto)
    df = criar_atributos(df)
    df = codificar(df)
    return aplicar_receita(df, receita)


historico_pronto = preparar(historico_bruto, receita)
semana_pronta = preparar(semana_bruta, receita)
semana_pronta.head()

,conta,gorjeta,sexo,fumante,turno,pessoas,percentual_gorjeta,conta_por_pessoa,fim_de_semana,grupo_grande,faixa_conta,dia_Fri,dia_Sat,dia_Sun,dia_Thur
200,-0.122987,4.00,0,1,0,3,0.824494,-0.567011,0,0,1,0,0,0,1
201,-0.793088,2.01,1,1,0,2,-0.078622,-0.522056,0,0,0,0,0,0,1
202,-0.763904,2.00,1,1,0,2,-0.141894,-0.478225,0,0,0,0,0,0,1
203,-0.382272,2.50,1,1,0,2,-0.164579,0.094951,0,0,1,0,0,0,1
204,0.081299,4.00,0,1,0,4,0.518946,-0.939294,0,1,1,0,0,0,1


**Sua vez 6.1:** escreva o código que confere o checklist de base pronta na `semana_pronta`: (1) não há valores nulos; (2) não há linhas duplicadas; (3) todas as colunas são numéricas; (4) as colunas são as mesmas do histórico. Imprima cada verificação com o resultado `True` ou `False`.

In [11]:
checagens = {
    'Sem valores nulos': not semana_pronta.isna().any().any(),
    'Sem duplicatas': not semana_pronta.duplicated().any(),
    'Todas as colunas numéricas': all(pd.api.types.is_numeric_dtype(t) for t in semana_pronta.dtypes),
    'Mesmo esquema do histórico': list(semana_pronta.columns) == list(historico_pronto.columns),
}
for descricao, passou in checagens.items():
    print(f'{descricao}: {bool(passou)}')
assert all(checagens.values())

Sem valores nulos: True
Sem duplicatas: True
Todas as colunas numéricas: True
Mesmo esquema do histórico: True


## 7. Entrega da base pronta
**Exemplo:** exportar a base pronta do histórico.

In [12]:
historico_pronto.to_csv("historico_pronto.csv", index=False)

**Sua vez 7.1:** exporte a `semana_pronta` para o arquivo `semana_pronta.csv`.

In [13]:
semana_pronta.to_csv('semana_pronta.csv', index=False)
print('semana_pronta.csv exportado:', semana_pronta.shape)

semana_pronta.csv exportado: (44, 15)


## 8. Deploy: um painel para o dono do restaurante
O dono não abre o Colab. A célula abaixo grava o arquivo `app.py`, um painel em Streamlit que recebe o CSV da semana, aplica parte do pipeline e mostra os resultados. O Colab não executa o painel diretamente; o arquivo é a base para o dashboard do projeto final.

**Exemplo:** painel com um indicador e a lista de mesas fora do padrão.

In [14]:
%%writefile app.py
from pathlib import Path
import pandas as pd
import streamlit as st

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"


def limpar(df):
    df = df.copy()
    df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
    return df.drop_duplicates()


def criar_atributos(df):
    df = df.copy()
    df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
    df["conta_por_pessoa"] = df["conta"] / df["pessoas"]
    return df


historico = criar_atributos(limpar(pd.read_csv(Path(__file__).parent / 'dados' / 'tips.csv').iloc[:200]))
receita = {"media": historico["percentual_gorjeta"].mean(), "desvio": historico["percentual_gorjeta"].std()}

st.title("Gorjetas da semana")
arquivo = st.file_uploader("Envie o CSV da semana", type="csv")
usar_exemplo = st.checkbox("Usar a semana de exemplo")

if arquivo is not None or usar_exemplo:
    dados = pd.read_csv(arquivo) if arquivo is not None else pd.read_csv(Path(__file__).parent / 'dados' / 'tips.csv').iloc[200:]
    semana = criar_atributos(limpar(dados))
    semana["percentual_z"] = (semana["percentual_gorjeta"] - receita["media"]) / receita["desvio"]
    st.metric("Percentual médio da semana", f"{semana['percentual_gorjeta'].mean():.1f}%",
              f"{semana['percentual_gorjeta'].mean() - receita['media']:.1f} p.p. em relação ao histórico")
    st.subheader("Mesas fora do padrão (mais de 2 desvios)")
    st.dataframe(semana[semana["percentual_z"].abs() > 2])
    st.subheader("Base da semana preparada")
    st.dataframe(semana)

Writing app.py


**Sua vez 8.1:** grave o arquivo `app_completo.py`, uma cópia do exemplo com mais dois elementos depois do primeiro indicador: o gasto médio por pessoa (`st.metric`) e um gráfico do percentual médio da gorjeta por dia (`st.bar_chart`).

In [15]:
%%writefile app_completo.py
from pathlib import Path
import pandas as pd
import streamlit as st

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"


def limpar(df):
    df = df.copy()
    df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
    return df.drop_duplicates()


def criar_atributos(df):
    df = df.copy()
    df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
    df["conta_por_pessoa"] = df["conta"] / df["pessoas"]
    return df


historico = criar_atributos(limpar(pd.read_csv(Path(__file__).parent / 'dados' / 'tips.csv').iloc[:200]))
receita = {"media": historico["percentual_gorjeta"].mean(), "desvio": historico["percentual_gorjeta"].std()}

st.title("Gorjetas da semana")
arquivo = st.file_uploader("Envie o CSV da semana", type="csv")
usar_exemplo = st.checkbox("Usar a semana de exemplo")

if arquivo is not None or usar_exemplo:
    dados = pd.read_csv(arquivo) if arquivo is not None else pd.read_csv(Path(__file__).parent / 'dados' / 'tips.csv').iloc[200:]
    semana = criar_atributos(limpar(dados))
    semana["percentual_z"] = (semana["percentual_gorjeta"] - receita["media"]) / receita["desvio"]
    st.metric("Percentual médio da semana", f"{semana['percentual_gorjeta'].mean():.1f}%",
              f"{semana['percentual_gorjeta'].mean() - receita['media']:.1f} p.p. em relação ao histórico")
    st.metric("Gasto médio por pessoa", f"US$ {semana['conta_por_pessoa'].mean():.2f}")
    st.subheader("Percentual médio da gorjeta por dia")
    por_dia = semana.groupby('dia')['percentual_gorjeta'].mean().reindex(['Thur', 'Fri', 'Sat', 'Sun']).dropna()
    st.bar_chart(por_dia)
    st.subheader("Mesas fora do padrão (mais de 2 desvios)")
    st.dataframe(semana[semana["percentual_z"].abs() > 2])
    st.subheader("Base da semana preparada")
    st.dataframe(semana)

Writing app_completo.py


**Pergunta 8.2:** que outro indicador você acrescentaria ao painel para ajudar o dono do restaurante? Qual pergunta de negócio ele responde?

**Resposta:**

Acrescentaria o número de mesas atendidas e o total de pessoas da semana. Eles respondem: a mudança no resultado ocorreu junto com uma mudança no volume de atendimento? O percentual médio sozinho não mostra se a equipe atendeu dez ou cem mesas. Compararia volume e percentual sem afirmar causalidade.

## Observações

A divisão 200/44 reproduz o exercício; não é uma amostra aleatória nem uma validação de modelo. A receita usa somente o histórico. O painel trabalha com valores originais para os indicadores, sem interpretar escores padronizados como dólares. A base original é preservada em `dados/tips.csv`.